# Notebook 09 — Validate and Tune Two Models

## Purpose
This notebook performs **validation and comparison** of the two selected  
classifiers for the DIP-based AI image detection pipeline using the normalized  
**25-dimensional Digital Image Processing (DIP) feature vectors**.

The notebook focuses on the two retained models from earlier classifier  
selection work:

- RBF Support Vector Machine (RBF SVM)
- Multi-Layer Perceptron (MLP)

Using the normalized training dataset, both models are evaluated with a  
consistent stratified cross-validation procedure so that their validation  
performance can be compared fairly before final independent test evaluation.

---

## Inputs
This notebook uses the normalized training dataset generated by:

- `06_Normalize_and_Prepare_Inputs.ipynb`

Expected input file:

- `metadata/vectors/train_feature_vectors_normalized.csv`

The dataset contains:

- Metadata columns:
  - `filename`
  - `class_label`
  - `source_dataset`
  - `subset`
- **25 normalized DIP feature columns**

The selected classifier hyperparameters are defined within this notebook based  
on the results of Notebook 07 and the finalized model-training design.

---

## Local Execution Assumptions
This notebook is designed to run within the **local GitHub project structure**  
or a compatible environment such as Google Colab.

It assumes:

- the project repository is available locally or cloned at runtime
- `src/project_config.py` is accessible
- prior pipeline notebooks have generated the normalized training dataset
- required Python packages (NumPy, pandas, scikit-learn) are installed

---

## Important Note
All features have already been normalized using a scaler fit only on the  
training dataset in Notebook 06. No additional normalization should be  
applied here.

This notebook uses only the **training split** and does **not** evaluate on  
the independent test dataset. Final test evaluation is reserved for  
Notebook 10.

---

## Process Overview
This notebook validates the two selected classifiers using a consistent  
cross-validation framework. The workflow includes:

1. loading and validating the normalized training dataset  
2. separating metadata from feature columns  
3. defining the selected classifier configurations  
4. initializing both classifiers with tuned hyperparameters  
5. performing stratified k-fold cross-validation  
6. evaluating models using multiple performance metrics  
7. comparing validation performance between the two models  
8. saving validation results for later reference  

---

## Outputs
This notebook produces the following files in `metadata/results/`:

- `cross_validation_results.csv`
- `classifier_comparison_tuned.csv`

These outputs summarize validation-stage model behavior and support  
interpretation of final test results generated later in Notebook 10.

---

## Key Design Choice
This notebook evaluates **both retained classifiers**, rather than narrowing  
the pipeline to a single model before final testing.

This design:

- preserves a fair comparison between different classifier types  
- uses identical validation procedures for both models  
- supports a stronger final comparison on the independent test set  

---

## Classifiers Evaluated
The classifiers evaluated in this notebook are:

- RBF Support Vector Machine (RBF SVM)
- Multi-Layer Perceptron (MLP)

Both models use tuned hyperparameters determined during earlier model  
selection work.

---

## Scope Limitation
This notebook does **not** perform:

- baseline multi-classifier selection across many candidate models  
- final training artifact generation  
- evaluation on the independent test dataset  
- confusion matrix or ROC generation on held-out test data  

These steps are handled in other notebooks.

---

## Cell-by-Cell Structure

### Cell 1
Import required libraries.

### Cell 2
Set up project paths and verify required input files.

### Cell 3
Load the normalized training dataset and preview its contents.

### Cell 4
Validate dataset integrity and prepare feature matrix (`X_train`) and encoded target vector (`y_train`).

### Cell 5
Define the selected classifier configurations.

### Cell 6
Run stratified cross-validation for both classifiers.

### Cell 7
Summarize and compare validation performance.

### Cell 8
Save validation outputs to `metadata/results/`.

In [1]:
# ============================================================
# Startup (Environment + Path Setup + Verification)
# ============================================================

import os
import sys
from pathlib import Path

# ------------------------------------------------------------
# Clone repo into Colab runtime (if needed)
# ------------------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# ------------------------------------------------------------
# Make src/ importable
# ------------------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ------------------------------------------------------------
# Import shared project configuration
# ------------------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_FILENAME,
    NUM_FEATURES,
    RANDOM_SEED,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    K_FOLDS,
    CV_SHUFFLE,
    CV_RANDOM_STATE,
)

# ------------------------------------------------------------
# Define project paths
# ------------------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
VECTORS_DIR = METADATA_ROOT / "vectors"
RESULTS_DIR = METADATA_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Define input and output paths
# ------------------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = VECTORS_DIR / TRAIN_NORMALIZED_FILENAME

CV_RESULTS_CSV_PATH = RESULTS_DIR / "cross_validation_results.csv"
TUNED_COMPARISON_CSV_PATH = RESULTS_DIR / "classifier_comparison_tuned.csv"

# ------------------------------------------------------------
# Verify required input files
# ------------------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    missing_files.append(str(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV))

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(missing_files)
    )

print("All required input files are present.")
print(f"Training data: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
print(f"Results dir:   {RESULTS_DIR}")



Cloning repository...
Verifying required input files...

All required input files are present.
Training data: /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Results dir:   /content/dip-ai-image-detection/metadata/results


In [2]:
# ============================================================
# Cell 1: Import Required Libraries
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder



In [3]:
# ============================================================
# Cell 2: Load Normalized Training Data
# ============================================================

print("Loading normalized training dataset...\n")

# ------------------------------------------------------------
# Load CSV
# ------------------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

# ------------------------------------------------------------
# Display basic dataset information
# ------------------------------------------------------------
print("Training dataset loaded successfully.\n")

print("Shape:")
print(df_train.shape)

print("\nColumn names:")
for col in df_train.columns:
    print(col)

# ------------------------------------------------------------
# Preview first few rows
# ------------------------------------------------------------
print("\nFirst 5 rows:")
display(df_train.head())



Loading normalized training dataset...

Training dataset loaded successfully.

Shape:
(14400, 29)

Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std

First 5 rows:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [4]:
# ============================================================
# Cell 3: Validate Training Data and Prepare Classifier Inputs
# ============================================================

print("Validating training dataset and preparing classifier inputs...\n")

# ------------------------------------------------------------
# Verify required metadata columns
# ------------------------------------------------------------
missing_metadata_cols = [col for col in METADATA_COLUMNS if col not in df_train.columns]

if missing_metadata_cols:
    raise ValueError(f"Missing required metadata columns: {missing_metadata_cols}")

print("Metadata columns verified.")

# ------------------------------------------------------------
# Identify feature columns
# ------------------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

print(f"Number of feature columns found: {len(feature_columns)}")

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

print("Feature count verified.")

# ------------------------------------------------------------
# Check for missing values
# ------------------------------------------------------------
if df_train[METADATA_COLUMNS + feature_columns].isnull().any().any():
    null_counts = df_train[METADATA_COLUMNS + feature_columns].isnull().sum()
    null_counts = null_counts[null_counts > 0]
    raise ValueError(f"Missing values detected:\n{null_counts}")

print("No missing values detected.")

# ------------------------------------------------------------
# Verify class labels
# ------------------------------------------------------------
unique_labels = sorted(df_train["class_label"].unique().tolist())
expected_labels = sorted([AI_LABEL, REAL_LABEL])

print(f"Observed class labels: {unique_labels}")

if unique_labels != expected_labels:
    raise ValueError(
        f"Expected class labels {expected_labels}, but found {unique_labels}."
    )

print("Class labels verified.")

# ------------------------------------------------------------
# Prepare feature matrix and target vector
# ------------------------------------------------------------
X_train = df_train[feature_columns].to_numpy()
y_labels = df_train["class_label"].to_numpy()

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_labels)

print("\nClassifier input arrays prepared successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# ------------------------------------------------------------
# Display encoded label mapping
# ------------------------------------------------------------
label_mapping = {
    class_name: int(label_encoder.transform([class_name])[0])
    for class_name in label_encoder.classes_
}

print("\nEncoded label mapping:")
for class_name, encoded_value in label_mapping.items():
    print(f"  {class_name} -> {encoded_value}")



Validating training dataset and preparing classifier inputs...

Metadata columns verified.
Number of feature columns found: 25
Feature count verified.
No missing values detected.
Observed class labels: ['ai', 'rl']
Class labels verified.

Classifier input arrays prepared successfully.
X_train shape: (14400, 25)
y_train shape: (14400,)

Encoded label mapping:
  ai -> 0
  rl -> 1


In [5]:
# ============================================================
# Cell 4: Define Selected Classifier Configurations
# ============================================================

print("Defining selected classifier configurations...\n")

# ------------------------------------------------------------
# Define the two retained classifier configurations
# ------------------------------------------------------------
classifier_configs = [
    {
        "classifier": "RBF SVM",
        "params": {
            "C": 100,
            "gamma": 0.01,
            "kernel": "rbf",
            "probability": True,
            "random_state": RANDOM_SEED
        }
    },
    {
        "classifier": "MLP",
        "params": {
            "hidden_layer_sizes": (64, 32),
            "alpha": 0.0001,
            "learning_rate_init": 0.001,
            "batch_size": 32,
            "max_iter": 500,
            "random_state": RANDOM_SEED
        }
    }
]

# ------------------------------------------------------------
# Display configured classifiers
# ------------------------------------------------------------
print(f"Number of classifiers configured: {len(classifier_configs)}\n")

for i, config in enumerate(classifier_configs, start=1):
    print(f"{i}. {config['classifier']}")
    print("   Parameters:")
    for param_name, param_value in config["params"].items():
        print(f"     {param_name} = {param_value}")
    print()



Defining selected classifier configurations...

Number of classifiers configured: 2

1. RBF SVM
   Parameters:
     C = 100
     gamma = 0.01
     kernel = rbf
     probability = True
     random_state = 42

2. MLP
   Parameters:
     hidden_layer_sizes = (64, 32)
     alpha = 0.0001
     learning_rate_init = 0.001
     batch_size = 32
     max_iter = 500
     random_state = 42



In [6]:
# ============================================================
# Cell 5: Run Stratified Cross-Validation for Both Classifiers
# ============================================================

print("Running stratified cross-validation for selected classifiers...\n")

# ------------------------------------------------------------
# Define cross-validation strategy and scoring metrics
# ------------------------------------------------------------
cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_list = []

# ------------------------------------------------------------
# Evaluate each classifier
# ------------------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"]

    print(f"Evaluating: {classifier_name}")

    if classifier_name == "RBF SVM":
        model = SVC(**params)
    elif classifier_name == "MLP":
        model = MLPClassifier(**params)
    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    result_row = {
        "classifier": classifier_name,
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "precision_mean": np.mean(scores["test_precision"]),
        "precision_std": np.std(scores["test_precision"]),
        "recall_mean": np.mean(scores["test_recall"]),
        "recall_std": np.std(scores["test_recall"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }

    cv_results_list.append(result_row)

    print(f"Completed: {classifier_name}")
    print(f"  Mean ROC-AUC: {result_row['roc_auc_mean']:.4f}")
    print(f"  Mean F1-score: {result_row['f1_mean']:.4f}\n")

print("Cross-validation complete.")



Running stratified cross-validation for selected classifiers...

Evaluating: RBF SVM
Completed: RBF SVM
  Mean ROC-AUC: 0.8792
  Mean F1-score: 0.8001

Evaluating: MLP
Completed: MLP
  Mean ROC-AUC: 0.8406
  Mean F1-score: 0.7629

Cross-validation complete.


In [7]:
# ============================================================
# Cell 6: Summarize and Compare Validation Performance
# ============================================================

print("Summarizing validation performance...\n")

# ------------------------------------------------------------
# Build comparison DataFrame
# ------------------------------------------------------------
df_cv_results = pd.DataFrame(cv_results_list)

# ------------------------------------------------------------
# Sort by primary metric
# ------------------------------------------------------------
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Create concise comparison summary table
# ------------------------------------------------------------
df_tuned_comparison = df_cv_results[
    [
        "classifier",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "roc_auc_mean"
    ]
].copy()

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------
print("Validation results ranked by ROC-AUC:\n")
display(df_cv_results)

print("\nCondensed tuned-model comparison:\n")
display(df_tuned_comparison)



Summarizing validation performance...

Validation results ranked by ROC-AUC:



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF SVM,0.798194,0.008415,0.792667,0.010336,0.807778,0.006044,0.800134,0.007684,0.879229,0.005399
1,MLP,0.762292,0.005915,0.761062,0.015354,0.766389,0.035431,0.762920,0.011760,0.840630,0.008315



Condensed tuned-model comparison:



,classifier,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
0,RBF SVM,0.798194,0.792667,0.807778,0.800134,0.879229
1,MLP,0.762292,0.761062,0.766389,0.762920,0.840630


In [8]:
# ============================================================
# Cell 7: Save Validation Outputs
# ============================================================

print("Saving validation outputs...\n")

# ------------------------------------------------------------
# Save full cross-validation results
# ------------------------------------------------------------
df_cv_results.to_csv(CV_RESULTS_CSV_PATH, index=False)
print(f"Saved cross-validation results: {CV_RESULTS_CSV_PATH}")

# ------------------------------------------------------------
# Save condensed tuned-model comparison
# ------------------------------------------------------------
df_tuned_comparison.to_csv(TUNED_COMPARISON_CSV_PATH, index=False)
print(f"Saved tuned comparison results: {TUNED_COMPARISON_CSV_PATH}")

print("\nValidation outputs saved successfully.")



Saving validation outputs...

Saved cross-validation results: /content/dip-ai-image-detection/metadata/results/cross_validation_results.csv
Saved tuned comparison results: /content/dip-ai-image-detection/metadata/results/classifier_comparison_tuned.csv

Validation outputs saved successfully.


In [9]:
# ============================================================
# Cell 8: Validation Output Sanity Check
# ============================================================

print("Performing sanity check on saved validation outputs...\n")

# ------------------------------------------------------------
# Reload saved files
# ------------------------------------------------------------
df_cv_check = pd.read_csv(CV_RESULTS_CSV_PATH)
df_tuned_check = pd.read_csv(TUNED_COMPARISON_CSV_PATH)

# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------
print("Cross-validation results shape:", df_cv_check.shape)
print("Tuned comparison shape:", df_tuned_check.shape)

# ------------------------------------------------------------
# Display first few rows
# ------------------------------------------------------------
print("\nCross-validation results preview:")
display(df_cv_check.head())

print("\nTuned comparison preview:")
display(df_tuned_check.head())

# ------------------------------------------------------------
# Basic validation checks
# ------------------------------------------------------------
expected_classifiers = {"RBF SVM", "MLP"}

cv_classifiers = set(df_cv_check["classifier"].unique())
tuned_classifiers = set(df_tuned_check["classifier"].unique())

if cv_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in CV results: {cv_classifiers}")

if tuned_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in tuned comparison: {tuned_classifiers}")

print("\nSanity check passed: expected classifiers present in both outputs.")
print("Validation notebook completed successfully.")



Performing sanity check on saved validation outputs...

Cross-validation results shape: (2, 11)
Tuned comparison shape: (2, 6)

Cross-validation results preview:


,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF SVM,0.798194,0.008415,0.792667,0.010336,0.807778,0.006044,0.800134,0.007684,0.879229,0.005399
1,MLP,0.762292,0.005915,0.761062,0.015354,0.766389,0.035431,0.762920,0.011760,0.840630,0.008315



Tuned comparison preview:


,classifier,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
0,RBF SVM,0.798194,0.792667,0.807778,0.800134,0.879229
1,MLP,0.762292,0.761062,0.766389,0.762920,0.840630



Sanity check passed: expected classifiers present in both outputs.
Validation notebook completed successfully.
